# Phase 00 | BaTiO₃ source provenance and scientific data architecture

**Research focus:** Source identity, reproducible registration, scientific evidence classification and chemistry-aware data engineering for BaTiO₃-based capacitor materials.

**Computational boundary:** Source registration and shallow container checks only. Published formation energies, DFT/VCA settings, crystal structures, machine-learning results and physical dielectric responses are not independently validated by Phase 00.

**Outputs:** `source_registry.csv`, `source_manifest.json`, `research_scope.md`, and `registration_findings.md`.

The workflow applies eight engineering qualities throughout: **clarity, simplicity, reproducibility, legibility, graceful error handling, evidential transparency, automated verification, and actionable findings.** These are implemented as measurable code and document properties rather than attributed or discussed as a personal framework.

## 1 | Scientific problem: linking composition, computations and evidence

**BaTiO₃ crystal chemistry.** BaTiO₃ is an oxide perovskite (ABO₃). The nominal A-site composition of the study is `(Ba₁₋ₓ₋ᵧSrₓLaᵧ)TiO₃`, with Ba, Sr and La assigned to the A site and Ti assigned to the B site. Nominal site occupancies require `x ≥ 0`, `y ≥ 0` and `x + y ≤ 1`.

**Physical significance.** Ba²⁺ → Sr²⁺ substitution is *isovalent*, whereas Ba²⁺ → La³⁺ is *aliovalent*. The latter raises a charge-compensation question. The published computational treatment uses VCA and electronic electron-counting procedures without explicit ionic vacancy defects in the specified calculations. A Ti coefficient appearing in a data table must therefore be interpreted from its method and schema, **not adjusted automatically to force a preferred formula**.

**Data-science connection.** A material-composition string is a scientific identifier, not just text. If composition is parsed incorrectly, subsequent descriptor values, material grouping and model predictions may be linked to the wrong chemistry. Phase 00 protects the published source bytes before any parsing changes the scientific record.

**Evidence hierarchy:** Published article → supplementary method definitions → computational workbook → chemically validated records → model-ready features → statistical predictions. A stage cannot be treated as verified solely because an earlier file opened successfully.

## 2 | Source schema and evidential roles

**Scientific interpretation.** The article describes an ML screen over 30,301 nominal compositions; supporting information defines computational formation-energy references and descriptor terminology. The associated workbook is the record-level source. These materials have **different evidential roles**: a paper explains a method, whereas a workbook stores values that must be checked against those definitions.

**Data-science implementation.** A fixed source catalogue separates `source_id`, file role, evidence class, DOI, provenance limitations and licensing status. Each observed file is then matched to its source specification. The source files are opened *read-only*. SHA-256 is calculated from bytes in bounded chunks, enabling a 25+ MB workbook to be fingerprinted without loading the entire spreadsheet into a DataFrame.

**Why this matters.** Without source IDs and stable byte fingerprints, two versions of a spreadsheet can be confused even if their visible names are nearly identical. A checksum establishes byte identity; it does **not** prove that a structure is stable or an energy calculation is accurate.

In [1]:
"""Phase 00 | BaTiO3 ceramic source provenance and deterministic integrity registry.

Scientific boundary:
    Container integrity and source identity, not a replication of DFT,
    a validation of formation energies, or a dielectric measurement.

Engineering properties:
    Standard-library implementation; bounded discovery; explicit ambiguity;
    streamed SHA-256; atomic, deterministic outputs; synthetic tests.
"""
from __future__ import annotations

import csv
import hashlib
import io
import json
import os
import re
import tempfile
import unicodedata
import unittest
import zipfile
from pathlib import Path

SCHEMA_VERSION = "3.0"
DOI_ARTICLE = "10.1063/5.0314819"
DOI_SUPPLEMENT = "10.60893/figshare.jap.c.8234602"

# Source roles are kept separate from scientific-evidence claims.
# Source metadata is not inferred from file-system timestamps.
SOURCE_CATALOG = (
    {
        "source_id": "SRC-001", "expected_filename": "Machine_Learning_First_Principles_Guided_Design_BaTiO3.pdf",
        "required": True, "role": "Peer-reviewed computational materials study",
        "evidence_class": "PUBLISHED_METHOD_AND_INTERPRETATION", "doi": DOI_ARTICLE,
        "scientific_relevance": "Crystal chemistry, doping design, DFT/VCA assumptions, reported ML screening",
        "provenance_limit": "Article content and scientific conclusions are reported, not independently reproduced",
        "licence_status": "Article states CC BY 4.0 except where otherwise noted; exact reuse terms require source review",
    },
    {
        "source_id": "SRC-002", "expected_filename": "Supplementary information.docx",
        "required": True, "role": "Published computational methods and feature definitions",
        "evidence_class": "SUPPLEMENTARY_SCIENTIFIC_DOCUMENTATION", "doi": DOI_SUPPLEMENT,
        "scientific_relevance": "Formation-energy reference formula, ML descriptor selection, methods and units",
        "provenance_limit": "Container registration does not establish the correctness of equations or calculations",
        "licence_status": "Supplementary file reuse terms not independently verified",
    },
    {
        "source_id": "SRC-003", "expected_filename": "Supplementary Dataset.xlsx",
        "required": True, "role": "Published computational workbook: sheets S1–S4",
        "evidence_class": "REPORTED_COMPUTATIONAL_DATA", "doi": DOI_SUPPLEMENT,
        "scientific_relevance": "Composition candidates, selected descriptors, published calculation and ML records",
        "provenance_limit": "No row-level chemistry, calculation or linkage audit in Phase 00",
        "licence_status": "Supplementary file reuse terms not independently verified",
    },
    {
        "source_id": "SRC-004", "expected_filename": "READ ME2(1).txt",
        "required": False, "role": "Project research roadmap, text-format draft",
        "evidence_class": "PROJECT_PLANNING_DOCUMENT", "doi": "",
        "scientific_relevance": "Scope and phase definitions; not a computational evidence source",
        "provenance_limit": "Project notes do not independently corroborate published calculations",
        "licence_status": "Not assessed",
    },
    {
        "source_id": "SRC-005", "expected_filename": "Pasted markdown.md",
        "required": False, "role": "Project research roadmap, Markdown draft",
        "evidence_class": "PROJECT_PLANNING_DOCUMENT", "doi": "",
        "scientific_relevance": "Scope and phase definitions; not a computational evidence source",
        "provenance_limit": "Project notes do not independently corroborate published calculations",
        "licence_status": "Not assessed",
    },
)

CSV_COLUMNS = (
    "source_id", "expected_filename", "observed_filename", "relative_location",
    "required", "file_status", "container_check", "size_bytes", "sha256",
    "source_role", "evidence_class", "doi", "acquisition_date", "licence_status",
    "scientific_relevance", "provenance_limit", "assessment", "next_action",
)


def file_sha256(path: Path, chunk_bytes: int = 1024 * 1024) -> str:
    """Streaming digest: the 25+ MB workbook is not loaded into RAM as one object."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(chunk_bytes), b""):
            digest.update(block)
    return digest.hexdigest()


def container_check(path: Path) -> str:
    """Shallow validation only: file signature / required OOXML member / readable text."""
    suffix = path.suffix.casefold()
    try:
        if suffix == ".pdf":
            with path.open("rb") as stream:
                return "PASS:PDF_SIGNATURE" if stream.read(5) == b"%PDF-" else "FAIL:PDF_SIGNATURE"
        if suffix in (".docx", ".xlsx"):
            if not zipfile.is_zipfile(path):
                return "FAIL:OFFICE_ZIP_CONTAINER"
            required_component = "word/document.xml" if suffix == ".docx" else "xl/workbook.xml"
            with zipfile.ZipFile(path) as archive:
                return ("PASS:OFFICE_STRUCTURE" if required_component in archive.namelist()
                        else "FAIL:OFFICE_COMPONENT")
        if suffix in (".md", ".txt"):
            data = path.read_bytes()
            if not data:
                return "FAIL:EMPTY_TEXT"
            (data.decode("utf-16") if data.startswith((b"\xff\xfe", b"\xfe\xff"))
             else data.decode("utf-8-sig"))
            return "PASS:TEXT_DECODING"
        return "FAIL:UNEXPECTED_EXTENSION"
    except (OSError, UnicodeError, zipfile.BadZipFile, ValueError, RuntimeError):
        return "FAIL:UNREADABLE_CONTAINER"



# Explicit configuration variables remain optional; no hidden external configuration file.
INPUT_DIRECTORY = None
SOURCE_OVERRIDES = {}
OUTPUT_DIRECTORY = None

## 3 | File discovery and evidence preservation

**Data-science implementation.** The search is deliberately bounded to the Phase 00 project directory and nearby `data/raw` folders. File discovery accepts harmless download-counter variations, including `Supplementary Dataset.xlsx` and `Supplementary Dataset(2).xlsx`, while checking extensions and matching normalized source names. Differing bytes under competing matching filenames produce **AMBIGUOUS_SOURCE**, not a silently selected dataset.

**Chemical significance.** Two similarly named workbook releases may carry different compositional coefficients, formation-energy labels or descriptor definitions. A silent substitution could manufacture apparent chemistry changes. Ambiguity is a scientifically meaningful failure state that prevents accidental mixing of evidence.

**Status semantics.** `REGISTERED_CONTAINER` means that the file exists and passes a basic PDF/Office/text format check. `MISSING_REQUIRED`, `INVALID_CONTAINER` or `AMBIGUOUS_SOURCE` mean that the source registry is incomplete. No such state is recoded as a successful scientific audit.

In [2]:
def filename_key(filename: str) -> str:
    """Normalise space, punctuation and trailing download counters: (1), (2), etc."""
    p = Path(unicodedata.normalize("NFKC", filename))
    stem = re.sub(r"\s*\(\d+\)$", "", p.stem.strip())
    return re.sub(r"[^a-z0-9]+", "", stem.casefold()) + p.suffix.casefold()


def source_directories(root: Path, input_directory: str | Path | None = None) -> list[Path]:
    """Deterministic bounded discovery, excluding generated files and environments."""
    root = Path(root).expanduser().resolve()
    seeds = [root, root / "data" / "raw"]
    if input_directory:
        entered = Path(input_directory).expanduser()
        entered = (entered if entered.is_absolute() else root / entered).resolve()
        seeds = [entered, entered / "data" / "raw", *seeds]
    excluded = {".venv", "venv", "env", ".git", "__pycache__", "site-packages",
                "phase00_outputs", "outputs", ".ipynb_checkpoints", "node_modules"}
    folders: list[Path] = []
    for folder in seeds:
        if folder.is_dir() and folder not in folders:
            folders.append(folder)
    if root.is_dir():
        for current, children, _ in os.walk(root):
            here = Path(current)
            depth = len(here.relative_to(root).parts)
            children[:] = sorted(c for c in children if depth < 3 and c not in excluded
                                 and not c.startswith("."))
            if here not in folders:
                folders.append(here)
            if len(folders) > 800:
                break
    return folders


def locate_source(spec: dict, folders: list[Path], overrides: dict | None = None,
                  root: Path | None = None) -> tuple[Path | None, str]:
    """Multiple matching but nonidentical source bytes are ambiguous, never silently chosen."""
    sid, expected = spec["source_id"], spec["expected_filename"]
    overrides = overrides or {}
    if sid in overrides:
        chosen = Path(overrides[sid]).expanduser()
        chosen = (chosen if chosen.is_absolute() else Path(root or Path.cwd()) / chosen).resolve()
        if not chosen.is_file() or chosen.suffix.casefold() != Path(expected).suffix.casefold():
            return None, "INVALID_OVERRIDE"
        return chosen, "OVERRIDE"

    key = filename_key(expected)
    matches: list[Path] = []
    for folder in folders:
        try:
            matches.extend(p.resolve() for p in folder.iterdir()
                           if p.is_file() and filename_key(p.name) == key)
        except (PermissionError, FileNotFoundError, OSError):
            continue
    matches = sorted(set(matches), key=lambda x: str(x).casefold())
    if not matches:
        return None, "NOT_FOUND"
    fingerprints = {file_sha256(p) for p in matches}
    if len(fingerprints) != 1:
        return None, "AMBIGUOUS_DIFFERENT_BYTES"
    # Stable choice: exact filename first, then the shortest path.
    selected = min(matches, key=lambda p: (p.name != expected, len(p.parts), str(p).casefold()))
    return selected, ("MATCHED_NAME" if selected.name == expected else "MATCHED_FILENAME_VARIANT")


def relative_location(path: Path, root: Path) -> str:
    try:
        return path.relative_to(root.resolve()).as_posix()
    except ValueError:
        # Absolute external directory is intentionally excluded from exported metadata.
        return "external_source"


def source_registration(root: Path, input_directory=None, overrides=None):
    root = Path(root).expanduser().resolve()
    folders = source_directories(root, input_directory)
    rows: list[dict] = []
    for spec in SOURCE_CATALOG:
        path, discovery = locate_source(spec, folders, overrides, root)
        required = spec["required"]
        if path is None:
            status = ("AMBIGUOUS_SOURCE" if discovery == "AMBIGUOUS_DIFFERENT_BYTES"
                      else "INVALID_SOURCE_OVERRIDE" if discovery == "INVALID_OVERRIDE"
                      else "MISSING_REQUIRED" if required else "OPTIONAL_NOT_PRESENT")
            check, size, fingerprint, observed, relative = "NOT_CHECKED", "", "", "", ""
            action = ("Resolve conflicting source copies using verified provenance" if status == "AMBIGUOUS_SOURCE"
                      else "Verify the explicitly selected file path and format" if status == "INVALID_SOURCE_OVERRIDE"
                      else "Source file required for a complete file-level registry" if required else "None")
        else:
            check = container_check(path)
            status = "REGISTERED_CONTAINER" if check.startswith("PASS:") else "INVALID_CONTAINER"
            size, fingerprint = path.stat().st_size, file_sha256(path)
            observed, relative = path.name, relative_location(path, root)
            action = ("None" if status == "REGISTERED_CONTAINER"
                      else "Inspect source integrity; preserve the original file for forensic comparison")
        rows.append({
            "source_id": spec["source_id"], "expected_filename": spec["expected_filename"],
            "observed_filename": observed, "relative_location": relative,
            "required": "yes" if required else "no", "file_status": status,
            "container_check": check, "size_bytes": size, "sha256": fingerprint,
            "source_role": spec["role"], "evidence_class": spec["evidence_class"],
            "doi": spec["doi"], "acquisition_date": "NOT_VERIFIED",
            "licence_status": spec["licence_status"],
            "scientific_relevance": spec["scientific_relevance"],
            "provenance_limit": spec["provenance_limit"],
            "assessment": discovery, "next_action": action,
        })
    complete = all(row["file_status"] == "REGISTERED_CONTAINER" for row in rows if row["required"] == "yes")
    return rows, ("COMPLETE_FILE_REGISTRATION" if complete else "INCOMPLETE_ACTION_REQUIRED")

## 4 | Stable exports and chemistry-aware scope constraints

**Data-science implementation.** The CSV establishes a one-row-per-source registry; JSON stores a machine-readable counterpart; Markdown records scientific scope and unresolved decisions. A fixed field order, sorted JSON keys, stable line endings, absence of execution timestamps and relative paths support repeatable exports. File writes are completed through a temporary file followed by an atomic replacement to avoid partially written reports.

**Chemistry linkage.** Published *formation energy* is defined with reference-state and stoichiometric terms in supplementary Text S1. The label is not interchangeable with total energy, dielectric constant or experimental loss. The supplement describes a 100-feature pool with 16 constants removed and two excluded variables (final energy and the formation-energy target), leaving 82 input features. This is a **reported filtering decision**; target-leakage verification belongs to later phases.

**Scientific caution.** Even an internally consistent value in eV/atom is conditional on the chosen reference states and computational approximations. Formation-energy screening does not establish thermodynamic stability against every competing phase, sinterability or device operation.

In [3]:
def research_scope(status: str) -> str:
    """Chemistry and data-science interpretation tied to the source evidence and audit boundary."""
    return f"""# Phase 00 — Scientific scope and provenance

**Research domain:** BaTiO₃-based perovskite dielectrics for capacitor applications  
**Status:** `{status}`  
**Evidence boundary:** File-level source registration; no independently reproduced calculations or measurements.

## 1. Scientific research problem

The study combines first-principles calculations with machine learning to examine how nominal Sr/La A-site substitution changes computed formation energies in BaTiO₃-related composition space. The reported screening space comprises 30,301 candidates. File registration establishes which published documents and workbook bytes are available before composition-level quality control.

**Chemistry:** BaTiO₃ is an ABO₃ perovskite: Ba at the A site, Ti at the B site and O in the oxide framework. The reported nominal composition is `(Ba₁₋ₓ₋ᵧSrₓLaᵧ)TiO₃`. Nonnegative A-site fractions require `x ≥ 0`, `y ≥ 0`, and `x + y ≤ 1`. These are compositional bookkeeping conditions, not proof of phase stability or synthesizability.

**Charge and defects:** Sr²⁺ replacing Ba²⁺ is isovalent; La³⁺ replacing Ba²⁺ is aliovalent. The article describes VCA electronic averaging and supercell electron-counting treatments, without explicitly adding ionic vacancy defects in these calculations. Real defects cannot be inferred from the nominal formula alone. The Ti–La coefficient interpretation remains an open chemistry-QC item until the workbook and method are reconciled.

## 2. Computational evidence and units

- **DFT/VCA reference data:** derived from electronic-structure approximations and calculation settings, not direct capacitor measurements.
- **Formation energy:** evaluated using explicit reference states and stoichiometric coefficients described in supplementary Text S1. Reported units must remain attached to each quantity (e.g., eV/atom); lower numerical energies are not, by themselves, a demonstration of equilibrium phase stability against every competing phase.
- **ML screening:** published model estimates are statistical surrogates, not additional independent DFT calculations. Method validation and grouping logic belong to later phases.
- **Experiment:** dielectric permittivity, loss tangent, frequency response and temperature dependence require measured specimens and protocols. They cannot be asserted from the workbook checksum.

## 3. Scientific metadata and descriptor lineage

Source roles are classified separately as research article, supporting method description, computational workbook and optional planning documents. The supplement reports an initial 100-feature set with 16 constant features removed, plus removal of a final-energy descriptor and the formation-energy target, leaving 82 model input descriptors. This is **reported feature selection**, not an independent reproduction of the calculation. Leakage audits must check that the energy target and proxy quantities do not enter the model input.

## 4. Provenance and reproducibility

- `SHA-256` identifies file bytes, not scientific truth or publisher authenticity.
- Container validation checks a PDF signature or required Office package member, not full scientific readability or cell integrity.
- Exact bytes and observed filenames are recorded; download suffixes such as `(1)` or `(2)` do not redefine chemical evidence.
- Generated CSV, JSON and Markdown use stable ordering, fixed schema and no execution timestamps or absolute account paths.
- Acquisition dates and supplementary reuse conditions remain unverified where source evidence is unavailable.

## 5. Transferable data-science logic

Data lineage in polymer glass-transition prediction, rheology and formulation studies illustrates the broader need to connect material identity, conditions and a labelled target. The **physical models do not transfer** from polymers to ferroelectric ceramics: glass-transition behaviour and viscoelastic moduli are distinct from perovskite lattice distortions, polarization and computational formation energies. The transferable method is preserving original evidence, units, material identity, model assumptions and applicability limits.

## 6. Scope boundaries and Phase 01 priorities

**Established at Phase 00:** Source-file locations, observed names, file sizes, SHA-256 identifiers, basic container checks and evidence classifications.

**Not established:** Correct S1–S4 row counts; unique composition keys; Ti/La occupancy interpretation; missingness; chemical constraints; descriptor leakage; DFT convergence; reliable ML generalisation; experimental dielectric performance.

**Next scientific checks:** Workbook worksheet structure; composition normalization and A-site balance; energy units and reference states; 231-record linkage reported in source documentation; features and target provenance; unresolved stoichiometric discrepancies. Any anomaly remains flagged until supported by source evidence.

## References (document identifiers, no personal attribution)

- Primary article: https://doi.org/{DOI_ARTICLE}
- Supplementary collection: https://doi.org/{DOI_SUPPLEMENT}
"""


def csv_text(rows: list[dict]) -> str:
    stream = io.StringIO(newline="")
    writer = csv.DictWriter(stream, fieldnames=CSV_COLUMNS, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue()


def atomic_write(target: Path, text: str) -> None:
    """Replace completed output files atomically; original scientific files are never opened for writing."""
    target.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", newline="\n",
                                     dir=target.parent, prefix="._phase00_", suffix=".tmp",
                                     delete=False) as stream:
        temp = Path(stream.name)
        stream.write(text)
    try:
        os.replace(temp, target)
    finally:
        if temp.exists():
            temp.unlink()


def write_outputs(root: Path, rows: list[dict], status: str, output_dir=None) -> dict[str, Path]:
    dest = Path(output_dir).expanduser().resolve() if output_dir else Path(root) / "phase00_outputs"
    registry = dest / "manifests" / "source_registry.csv"
    manifest = dest / "manifests" / "source_manifest.json"
    scope = dest / "docs" / "research_scope.md"
    report = dest / "reports" / "registration_findings.md"
    source_records = [
        {
            "source_id": r["source_id"], "expected_filename": r["expected_filename"],
            "observed_filename": r["observed_filename"] or None,
            "relative_location": r["relative_location"] or None,
            "required": r["required"] == "yes", "status": r["file_status"],
            "format_check": r["container_check"],
            "size_bytes": r["size_bytes"] if r["size_bytes"] != "" else None,
            "sha256": r["sha256"] or None,
            "evidence_class": r["evidence_class"], "doi": r["doi"] or None,
            "acquisition_date": r["acquisition_date"], "licence_status": r["licence_status"],
            "next_action": r["next_action"],
        }
        for r in rows
    ]
    manifest_data = {
        "schema_version": SCHEMA_VERSION, "phase": "00", "status": status,
        "scientific_validation_status": "NOT_PERFORMED", "checksum_algorithm": "SHA-256",
        "record_count": len(source_records), "sources": source_records,
    }
    findings = [
        f"| {r['source_id']} | {r['file_status']} | {r['assessment']} | {r['next_action']} |"
        for r in rows
    ]
    report_text = (
        "# Phase 00 — Source registration findings\n\n"
        f"**Status:** `{status}`  \n**Scientific validation:** Not performed  \n"
        "**Interpretation:** Source-container checks and cryptographic byte identities only.\n\n"
        "| Source | Result | Discovery assessment | Disposition |\n"
        "|---|---|---|---|\n" + "\n".join(findings) + "\n\n"
        "## Scientific decisions retained for subsequent phases\n\n"
        "- Verify workbook S1–S4 structure and reported candidate/labelled populations.\n"
        "- Reconcile nominal A-site composition constraints with Ti/La coefficients before normalization.\n"
        "- Trace formation-energy labels to reference-state definitions and computational settings.\n"
        "- Distinguish calculated labels, model predictions and any physical measurements.\n"
        "- Audit descriptor provenance and possible target leakage.\n"
        "- Treat absent or ambiguous inputs as incomplete registration, not a successful audit.\n"
    )
    content = {
        registry: csv_text(rows),
        manifest: json.dumps(manifest_data, sort_keys=True, ensure_ascii=False, indent=2) + "\n",
        scope: research_scope(status), report: report_text,
    }
    for path, text in content.items():
        atomic_write(path, text)
    with registry.open("r", encoding="utf-8", newline="") as stream:
        parsed = list(csv.DictReader(stream))
    decoded = json.loads(manifest.read_text(encoding="utf-8"))
    if len(parsed) != len(rows) or decoded["record_count"] != len(rows):
        raise RuntimeError("OUTPUT_VERIFICATION_FAILED: source counts inconsistent")
    if [r["sha256"] for r in parsed] != [r["sha256"] for r in rows]:
        raise RuntimeError("OUTPUT_VERIFICATION_FAILED: hash identities inconsistent")
    return {"source_registry": registry, "source_manifest": manifest,
            "research_scope": scope, "registration_findings": report}


def run_phase00(root=None, input_directory=None, overrides=None, output_directory=None):
    root = Path(root or Path.cwd()).expanduser().resolve()
    rows, status = source_registration(root, input_directory, overrides)
    outputs = write_outputs(root, rows, status, output_directory)
    return {"status": status, "records": rows, "outputs": outputs,
            "source_count": len(rows), "required_registered": sum(
                r["required"] == "yes" and r["file_status"] == "REGISTERED_CONTAINER" for r in rows)}


def print_summary(result: dict) -> None:
    """Concise, action-oriented assessment without absolute filesystem or account identifiers."""
    print("PHASE 00 |", result["status"])
    print("Required scientific files registered:", f"{result['required_registered']}/3")
    for row in result["records"]:
        print(f"{row['source_id']} | {row['file_status']} | "
              f"{row['observed_filename'] or row['expected_filename']}")
        if row["next_action"] != "None":
            print("  Assessment:", row["next_action"])
    print("Output files:", ", ".join(path.name for path in result["outputs"].values()))
    print("Scientific-data validation: NOT_PERFORMED (reserved for later phases)")

## 5 | Phase 00 registration result

**Engineering invariant.** The required three sources—article, supplementary methodology and computational workbook—must all be found and pass shallow format checks for `COMPLETE_FILE_REGISTRATION`. Optional planning documents are recorded when available but do not affect this status.

**Evidential boundary.** The status means **file registration is complete**, not that S1–S4, the claimed 231 labelled records, charge compensation, selected features or reported ML performance have been independently validated.

The result includes structured findings for missing, corrupted or conflicting sources; unknown acquisition dates and reuse terms remain explicitly unverified.

In [4]:
result = run_phase00(
    root=Path.cwd(),
    input_directory=INPUT_DIRECTORY,
    overrides=SOURCE_OVERRIDES,
    output_directory=OUTPUT_DIRECTORY,
)
print_summary(result)

PHASE 00 | COMPLETE_FILE_REGISTRATION
Required scientific files registered: 3/3
SRC-001 | REGISTERED_CONTAINER | Machine_Learning_First_Principles_Guided_Design_BaTiO3.pdf
SRC-002 | REGISTERED_CONTAINER | Supplementary information.docx
SRC-003 | REGISTERED_CONTAINER | Supplementary Dataset.xlsx
SRC-004 | OPTIONAL_NOT_PRESENT | READ ME2(1).txt
SRC-005 | OPTIONAL_NOT_PRESENT | Pasted markdown.md
Output files: source_registry.csv, source_manifest.json, research_scope.md, registration_findings.md
Scientific-data validation: NOT_PERFORMED (reserved for later phases)


## 6 | Automated engineering tests and deterministic reruns

**Tests assess program behaviour rather than the scientific validity of the published values.** Synthetic miniature PDF and Office files isolate SHA-256 consistency, download-suffix normalization, wrong extensions, malformed containers, UTF-16 text support, missing inputs, conflicting copies, nested-folder discovery, unmodified original files, stable outputs and exclusion of absolute filesystem paths.

**Reproducibility criterion.** Repeating Phase 00 with unchanged input bytes and identical project-relative filenames must reproduce byte-identical CSV, JSON and Markdown outputs. The tests distinguish repeatable provenance documents from the much harder task of reproducing published DFT energies or generalizable dielectric predictions.

In [5]:
def run_engineering_tests() -> unittest.result.TestResult:
    """Synthetic fixtures test engineering behaviour without altering published materials files."""
    class TestRegistry(unittest.TestCase):
        def test_sha256_reference(self):
            with tempfile.TemporaryDirectory() as td:
                p = Path(td) / "test.bin"; p.write_bytes(b"abc")
                self.assertEqual(file_sha256(p), hashlib.sha256(b"abc").hexdigest())

        def test_download_suffix_normalization(self):
            name = SOURCE_CATALOG[2]["expected_filename"]
            self.assertEqual(filename_key(name), filename_key("Supplementary Dataset(2).xlsx"))
            self.assertEqual(filename_key(name), filename_key("Supplementary_Dataset (19).xlsx"))

        def test_wrong_extension_not_match(self):
            self.assertNotEqual(filename_key("Supplementary Dataset.xlsx"), filename_key("Supplementary Dataset.csv"))

        def test_pdf_bad_header(self):
            with tempfile.TemporaryDirectory() as td:
                p = Path(td) / "x.pdf"; p.write_bytes(b"not PDF")
                self.assertEqual(container_check(p), "FAIL:PDF_SIGNATURE")

        def test_pdf_valid_header(self):
            with tempfile.TemporaryDirectory() as td:
                p = Path(td) / "x.pdf"; p.write_bytes(b"%PDF-1.7\n")
                self.assertEqual(container_check(p), "PASS:PDF_SIGNATURE")

        def test_office_malformed_zip(self):
            with tempfile.TemporaryDirectory() as td:
                p = Path(td) / "x.xlsx"; p.write_bytes(b"not a zip")
                self.assertEqual(container_check(p), "FAIL:OFFICE_ZIP_CONTAINER")

        def test_office_required_component(self):
            with tempfile.TemporaryDirectory() as td:
                p = Path(td) / "x.xlsx"
                with zipfile.ZipFile(p, "w") as archive:
                    archive.writestr("xl/workbook.xml", "<workbook/>")
                self.assertEqual(container_check(p), "PASS:OFFICE_STRUCTURE")

        def test_utf16_text_accepted(self):
            with tempfile.TemporaryDirectory() as td:
                p = Path(td) / "note.txt"; p.write_bytes("hello".encode("utf-16"))
                self.assertEqual(container_check(p), "PASS:TEXT_DECODING")

        def test_missing_files_report_incomplete(self):
            with tempfile.TemporaryDirectory() as td:
                result = run_phase00(td)
                self.assertEqual(result["status"], "INCOMPLETE_ACTION_REQUIRED")
                self.assertTrue(result["outputs"]["source_manifest"].is_file())

        def test_filenames_without_suffix(self):
            with tempfile.TemporaryDirectory() as td:
                root = Path(td)
                self._make_three(root, False)
                self.assertEqual(run_phase00(root)["status"], "COMPLETE_FILE_REGISTRATION")

        def test_filenames_with_suffix(self):
            with tempfile.TemporaryDirectory() as td:
                root = Path(td)
                self._make_three(root, True)
                self.assertEqual(run_phase00(root)["status"], "COMPLETE_FILE_REGISTRATION")

        def test_nested_raw_discovery(self):
            with tempfile.TemporaryDirectory() as td:
                root = Path(td); p = root / "data" / "raw"; p.mkdir(parents=True)
                self._make_three(p, True)
                self.assertEqual(run_phase00(root)["status"], "COMPLETE_FILE_REGISTRATION")

        def test_conflicting_sources_ambiguous(self):
            with tempfile.TemporaryDirectory() as td:
                root = Path(td); p = root / "data" / "raw"; p.mkdir(parents=True)
                name = SOURCE_CATALOG[0]["expected_filename"]
                (root / name).write_bytes(b"%PDF-first")
                (p / name).write_bytes(b"%PDF-second")
                _, discovery = locate_source(SOURCE_CATALOG[0], source_directories(root))
                self.assertEqual(discovery, "AMBIGUOUS_DIFFERENT_BYTES")

        def test_output_idempotence(self):
            with tempfile.TemporaryDirectory() as td:
                root = Path(td); self._make_three(root, True)
                first = run_phase00(root)
                a = {k: p.read_bytes() for k, p in first["outputs"].items()}
                second = run_phase00(root)
                b = {k: p.read_bytes() for k, p in second["outputs"].items()}
                self.assertEqual(a, b)

        def test_raw_bytes_unchanged(self):
            with tempfile.TemporaryDirectory() as td:
                root = Path(td); self._make_three(root, False)
                original = {p.name: p.read_bytes() for p in root.iterdir() if p.is_file()}
                run_phase00(root)
                self.assertEqual(original, {p.name: p.read_bytes() for p in root.iterdir() if p.is_file()})

        def test_no_absolute_path_exported(self):
            with tempfile.TemporaryDirectory() as td:
                root = Path(td); self._make_three(root, False)
                result = run_phase00(root)
                body = "\n".join(p.read_text(encoding="utf-8") for p in result["outputs"].values())
                self.assertNotIn(str(root), body)

        @staticmethod
        def _make_three(root, with_suffix):
            names = (["Machine_Learning_First_Principles_Guided_Design_BaTiO3(1).pdf",
                      "Supplementary information(2).docx", "Supplementary Dataset(2).xlsx"]
                     if with_suffix else [x["expected_filename"] for x in SOURCE_CATALOG[:3]])
            for name in names:
                p = root / name
                if p.suffix == ".pdf":
                    p.write_bytes(b"%PDF-1.7\n")
                else:
                    with zipfile.ZipFile(p, "w") as archive:
                        archive.writestr("word/document.xml" if p.suffix == ".docx" else "xl/workbook.xml", "<test/>")

    stream = io.StringIO()
    test_result = unittest.TextTestRunner(stream=stream, verbosity=1).run(
        unittest.defaultTestLoader.loadTestsFromTestCase(TestRegistry))
    print(stream.getvalue().strip())
    if not test_result.wasSuccessful():
        raise RuntimeError("ENGINEERING_TESTS_FAILED")
    return test_result

In [6]:
engineering_tests = run_engineering_tests()
first_bytes = {name: path.read_bytes() for name, path in result["outputs"].items()}
second_result = run_phase00(
    root=Path.cwd(),
    input_directory=INPUT_DIRECTORY,
    overrides=SOURCE_OVERRIDES,
    output_directory=OUTPUT_DIRECTORY,
)
second_bytes = {name: path.read_bytes() for name, path in second_result["outputs"].items()}
assert first_bytes == second_bytes, "REPRODUCIBILITY_FAILURE: exported bytes changed"
assert result["status"] == second_result["status"], "STATUS_CHANGED_WITHOUT_INPUT_CHANGE"
print("REPEAT_EXECUTION: PASS | byte-identical output documents")
print("SOURCE_REGISTRATION:", second_result["status"])
print("ENGINEERING_TESTS:", engineering_tests.testsRun, "passed")

................
----------------------------------------------------------------------
Ran 16 tests in 0.246s

OK
REPEAT_EXECUTION: PASS | byte-identical output documents
SOURCE_REGISTRATION: COMPLETE_FILE_REGISTRATION
ENGINEERING_TESTS: 16 passed


## 7 | Scientific decisions retained for Phase 01

| Research question | Chemistry rationale | Data-science validation |
|---|---|---|
| Are S1–S4 records correctly identified? | Different sheets may encode candidates, computed labels and feature views | Sheet inventory, dtype audit, row/column counts, stable record keys |
| Do composition fields represent the same nominal formula? | A-site Sr/La substitution and Ti-site coefficients require method-aware interpretation | Non-destructive parsing, occupancy rules, anomaly flags, source-cell provenance |
| Are energy quantities correctly labelled? | Total energy and formation energy differ; units and reference states matter | Unit dictionary, target mapping, equation/source cross-reference |
| Are computational and predictive values distinguishable? | ML-screened energies are not independent DFT results | `evidence_type` field, training/holdout linkage, prediction provenance |
| Are model features scientifically defensible? | Descriptors must represent the input composition, not leak target energy | Constant-feature review, target-proxy screening, subgroup holdouts |
| Do results imply experimental device performance? | Dielectric loss and temperature stability require material processing and measurement | Explicit separation of reported computation, model inference and experiment |

**Materials-informatics principle:** Traceability and error visibility create a reliable foundation for later scientific inference; they do not substitute for physics or chemistry validation. The generated reports retain verified findings, unverified metadata and follow-up questions without overstating conclusions.

## 8 | Source references and phase boundary

- Research article DOI: `10.1063/5.0314819`.
- Supplementary collection DOI: `10.60893/figshare.jap.c.8234602`.

**Phase 00 evidence:** local file existence, byte identity, shallow format and documented roles.  
**Phase 01 onward:** chemistry-aware sheet contents, compositional constraints, source-to-record linkage, energy units and model readiness.